In [1]:
!pip install torch torchvision torchaudio
!pip install transformers datasets
!pip install easyocr
!pip install python-docx
!pip install opencv-python
!pip install pandas
!pip install pillow
!pip install fastapi uvicorn
!pip install seqeval

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 32.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 978.2/978.2 kB 34.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 299.6/299.6 kB 16.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 4.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for seqeval: filename=seqeval-1.2.2-py3-none-any.whl size=16162 sha256=7443c8b9a77c4c1c0855d0c511dcb9c0b5abf627536b8d19c653d27beae2838e
  Stored in directory: /root/.cache/pip/wheels/5f/b8/73/0b2c1a76b701a677653dd79ece07cfabd7457989dbfbdcd8d7
Successfully built seqeval


In [2]:
import os
import cv2
import torch
import pandas as pd
import numpy as np

from PIL import Image

import easyocr

from docx import Document

from transformers import LayoutLMv3Processor
from transformers import LayoutLMv3ForTokenClassification

In [4]:
ROOT = "data"

TRAIN_FOLDER = os.path.join(ROOT,"train")
TEST_FOLDER = os.path.join(ROOT,"test")

TRAIN_CSV = os.path.join(ROOT,"train.csv")
TEST_CSV = os.path.join(ROOT,"test.csv")

print(TRAIN_FOLDER)
print(TEST_FOLDER)

print(TRAIN_CSV)
print(TEST_CSV)

data/train
data/test
data/train.csv
data/test.csv


In [6]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [13]:
import os

ROOT = "/content/drive/MyDrive/"

TRAIN_FOLDER = os.path.join(ROOT, "train")
TEST_FOLDER = os.path.join(ROOT, "test")

TRAIN_CSV = os.path.join(ROOT, "train.csv")
TEST_CSV = os.path.join(ROOT, "test.csv")

print(TRAIN_FOLDER)
print(TEST_FOLDER)
print(TRAIN_CSV)
print(TEST_CSV)

/content/drive/MyDrive/train
/content/drive/MyDrive/test
/content/drive/MyDrive/train.csv
/content/drive/MyDrive/test.csv


In [14]:
import pandas as pd

train_df = pd.read_csv(TRAIN_CSV)
test_df = pd.read_csv(TEST_CSV)

In [15]:
print("Training Dataset")
display(train_df.head())

print("\nTesting Dataset")
display(test_df.head())

Training Dataset


,File Name,Aggrement Value,Aggrement Start Date,Aggrement End Date,Renewal Notice (Days),Party One,Party Two
0,6683127-House-Rental-Contract-GERALDINE-GALINA...,6500,20.05.2007,20.05.2008,15.0,"Antonio Levy S. Ingles, Jr. and/or Mary Rose C...",GERALDINE Q. GALINATO
1,6683129-House-Rental-Contract-Geraldine-Galina...,6500,20.05.2007,20.05.2008,15.0,"Antonio Levy S. Ingles, Jr. and/or Mary Rose C...",GERALDINE Q. GALINATO
2,18325926-Rental-Agreement-1,4000,05.12.2008,31.11.2009,90.0,MR.K.Kuttan,P.M. Narayana Namboodri
3,24158401-Rental-Agreement,12000,01.04.2008,31.03.2009,60.0,Hanumaiah,Vishal Bhardwaj
4,36199312-Rental-Agreement,3800,01.05.2010,31.04.2011,30.0,Balaji.R,Kartheek R



Testing Dataset


,File Name,Aggrement Value,Aggrement Start Date,Aggrement End Date,Renewal Notice (Days),Party One,Party Two
0,24158401-Rental-Agreement,12000,01.04.2008,31.03.2009,60,Hanumaiah,Vishal Bhardwaj
1,95980236-Rental-Agreement,9000,01.04.2010,31.03.2011,30,S.Sakunthala,V.V.Ravi Kian
2,156155545-Rental-Agreement-Kns-Home,12000,15.12.2012,14.11.2013,30,V.K.NATARAJ,VYSHNAVI DAIRY SPECIALITIES Private Ltd
3,228094620-Rental-Agreement,15000,07.07.2013,06.06.2014,30,KAPIL MEHROTRA,.B.Kishore


In [16]:
print("Training Shape :", train_df.shape)
print("Testing Shape  :", test_df.shape)

print("\nTraining Columns")
print(train_df.columns.tolist())

Training Shape : (10, 7)
Testing Shape  : (4, 7)

Training Columns
['File Name', 'Aggrement Value', 'Aggrement Start Date', 'Aggrement End Date', 'Renewal Notice (Days)', 'Party One', 'Party Two']


In [17]:
train_files = os.listdir(TRAIN_FOLDER)

print("Total Training Files :", len(train_files))

for file in train_files:
    print(file)

Total Training Files : 10
6683129-House-Rental-Contract-Geraldine-Galinato-v2.docx
54770958-Rental-Agreement.png
6683127-House-Rental-Contract-GERALDINE-GALINATO-v2-Page-1.docx
47854715-RENTAL-AGREEMENT.docx
50070534-RENTAL-AGREEMENT.docx
18325926-Rental-Agreement-1.docx
44737744-Maddireddy-Bhargava-Reddy-Rental-Agreement.docx
36199312-Rental-Agreement.png
46239065-Standard-Rental-Agreement-Rental-With-Performance-Fee.docx
54945838-Rental-Agreement.png


In [18]:
test_files = os.listdir(TEST_FOLDER)

print("Total Test Files :", len(test_files))

for file in test_files:
    print(file)

Total Test Files : 4
228094620-Rental-Agreement.pdf.docx
156155545-Rental-Agreement-Kns-Home.pdf.docx
95980236-Rental-Agreement.png
24158401-Rental-Agreement.png


In [19]:
import easyocr

reader = easyocr.Reader(['en'], gpu=False)

Progress: |██████████████████████████████████████████████████| 100.0% Complete

Progress: |██████████████████████████████████████████████████| 100.0% Complete

In [20]:
from docx import Document
from PIL import Image
import os

In [21]:
def extract_docx_text(file_path):

    doc = Document(file_path)

    text = ""

    for para in doc.paragraphs:
        text += para.text + "\n"

    return text

In [22]:
def extract_png_text(file_path):

    result = reader.readtext(file_path, detail=0)

    text = "\n".join(result)

    return text

In [23]:
def read_document(file_path):

    extension = os.path.splitext(file_path)[1].lower()

    if extension == ".docx":
        return extract_docx_text(file_path)

    elif extension == ".png":
        return extract_png_text(file_path)

    else:
        return ""

In [24]:
sample_file = os.path.join(TRAIN_FOLDER, train_files[0])

print(sample_file)

text = read_document(sample_file)

print(text[:3000])

/content/drive/MyDrive/train/6683129-House-Rental-Contract-Geraldine-Galinato-v2.docx





House Rental Contract
KNOWN ALL MEN BY THESE PRESENTS:
This House Rental Contract, made and entered into this 20th day of May 2007 at Manila by and between:
Antonio Levy S. Ingles. Jr. and/or Mary Rose C. Ingles, of legal age, with residence and postal address at Unit 2006 EGI Taft Tower 2339 Taft Avenue, Malate, Manila, And herein referred to as the Owner(s),
— And —
GERALDINE O. GALINATO of legal age, with residence and postal address at 6 Manganese Road, Pilar Village, Las Pinas, Metro Manila, And herein referred to as the
Resident(s),
WITNESSETH:
In consideration of the agreements of the Resident(s), known as: GERALDINE O. GALINATO. the Owner(s), known as: Antonio Levy S. Ingles. Jr. and/or Mary Rose C. Ingles, hereby rent their the dwelling/house located at Lot 6, Block 20, Royal South Townhomes, Marcos Alvarez Avenue, Talon 5, Las Pinas City, Metro Manila for the period commencing on the 20

In [25]:
documents = []

for file in train_files:

    path = os.path.join(TRAIN_FOLDER, file)

    text = read_document(path)

    documents.append({
        "filename": file,
        "text": text
    })

documents_df = pd.DataFrame(documents)

documents_df.head()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


,filename,text
0,6683129-House-Rental-Contract-Geraldine-Galina...,\n\n\n\n\nHouse Rental Contract\nKNOWN ALL MEN...
1,54770958-Rental-Agreement.png,RENTAL AGREEMENT\nThis RENTA\nAGREEMENT\nmade\...
2,6683127-House-Rental-Contract-GERALDINE-GALINA...,House Rental Contract\nKNOWN ALL MEN BY THESE ...
3,47854715-RENTAL-AGREEMENT.docx,\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nRENTAL AGR...
4,50070534-RENTAL-AGREEMENT.docx,\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n...


In [27]:
train_df.rename(
    columns={train_df.columns[0]: "filename"},
    inplace=True
)

dataset = train_df.merge(
    documents_df,
    on="filename",
    how="left"
)

dataset.head()

,filename,Aggrement Value,Aggrement Start Date,Aggrement End Date,Renewal Notice (Days),Party One,Party Two,text
0,6683127-House-Rental-Contract-GERALDINE-GALINA...,6500,20.05.2007,20.05.2008,15.0,"Antonio Levy S. Ingles, Jr. and/or Mary Rose C...",GERALDINE Q. GALINATO,NaN
1,6683129-House-Rental-Contract-Geraldine-Galina...,6500,20.05.2007,20.05.2008,15.0,"Antonio Levy S. Ingles, Jr. and/or Mary Rose C...",GERALDINE Q. GALINATO,NaN
2,18325926-Rental-Agreement-1,4000,05.12.2008,31.11.2009,90.0,MR.K.Kuttan,P.M. Narayana Namboodri,NaN
3,24158401-Rental-Agreement,12000,01.04.2008,31.03.2009,60.0,Hanumaiah,Vishal Bhardwaj,NaN
4,36199312-Rental-Agreement,3800,01.05.2010,31.04.2011,30.0,Balaji.R,Kartheek R,NaN


In [28]:
!pip install -q transformers datasets accelerate seqeval

In [29]:
import torch
from transformers import (
    LayoutLMv3Processor,
    LayoutLMv3ForTokenClassification,
    TrainingArguments,
    Trainer
)

from datasets import Dataset

In [30]:
processor = LayoutLMv3Processor.from_pretrained(
    "microsoft/layoutlmv3-base",
    apply_ocr=False
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/275 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/856 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.14k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

In [31]:
label2id = {
    "O":0,

    "B-START_DATE":1,
    "I-START_DATE":2,

    "B-END_DATE":3,
    "I-END_DATE":4,

    "B-NOTICE":5,
    "I-NOTICE":6,

    "B-PARTY1":7,
    "I-PARTY1":8,

    "B-PARTY2":9,
    "I-PARTY2":10
}

id2label = {v:k for k,v in label2id.items()}

In [32]:
model = LayoutLMv3ForTokenClassification.from_pretrained(
    "microsoft/layoutlmv3-base",
    num_labels=len(label2id),
    label2id=label2id,
    id2label=id2label
)

model.safetensors:   0%|          | 0.00/501M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/212 [00:00<?, ?it/s]

[transformers] LayoutLMv3ForTokenClassification LOAD REPORT from: microsoft/layoutlmv3-base
Key                        | Status  | 
---------------------------+---------+-
classifier.dense.weight    | MISSING | 
classifier.dense.bias      | MISSING | 
classifier.out_proj.bias   | MISSING | 
classifier.out_proj.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [33]:
dataset.head()

,filename,Aggrement Value,Aggrement Start Date,Aggrement End Date,Renewal Notice (Days),Party One,Party Two,text
0,6683127-House-Rental-Contract-GERALDINE-GALINA...,6500,20.05.2007,20.05.2008,15.0,"Antonio Levy S. Ingles, Jr. and/or Mary Rose C...",GERALDINE Q. GALINATO,NaN
1,6683129-House-Rental-Contract-Geraldine-Galina...,6500,20.05.2007,20.05.2008,15.0,"Antonio Levy S. Ingles, Jr. and/or Mary Rose C...",GERALDINE Q. GALINATO,NaN
2,18325926-Rental-Agreement-1,4000,05.12.2008,31.11.2009,90.0,MR.K.Kuttan,P.M. Narayana Namboodri,NaN
3,24158401-Rental-Agreement,12000,01.04.2008,31.03.2009,60.0,Hanumaiah,Vishal Bhardwaj,NaN
4,36199312-Rental-Agreement,3800,01.05.2010,31.04.2011,30.0,Balaji.R,Kartheek R,NaN


In [35]:
# Find the actual column names
print(dataset.columns.tolist())

# Create a mapping automatically
column_mapping = {}

for col in dataset.columns:
    c = col.lower()

    if "start" in c and "date" in c:
        column_mapping["start"] = col

    elif "end" in c and "date" in c:
        column_mapping["end"] = col

    elif "renew" in c:
        column_mapping["notice"] = col

    elif "party" in c and ("one" in c or "1" in c or "first" in c):
        column_mapping["party1"] = col

    elif "party" in c and ("two" in c or "2" in c or "second" in c):
        column_mapping["party2"] = col

print(column_mapping)

['filename', 'Aggrement Value', 'Aggrement Start Date', 'Aggrement End Date', 'Renewal Notice (Days)', 'Party One', 'Party Two', 'text']
{'start': 'Aggrement Start Date', 'end': 'Aggrement End Date', 'notice': 'Renewal Notice (Days)', 'party1': 'Party One', 'party2': 'Party Two'}


In [44]:
training_data = []

questions = {
    "start": "What is the agreement start date?",
    "end": "What is the agreement end date?",
    "notice": "What is the renewal notice period?",
    "party1": "Who is party one?",
    "party2": "Who is party two?"
}

for _, row in dataset.iterrows():

    context = row["text"]

    for key, question in questions.items():

        if key not in column_mapping:
            continue

        answer = str(row[column_mapping[key]])

        training_data.append({
            "context": context,
            "question": question,
            "answer": answer
        })

training_df = pd.DataFrame(training_data)

display(training_df.head())

,context,question,answer
0,House Rental Contract\nKNOWN ALL MEN BY THESE ...,What is the agreement start date?,20.05.2007
1,House Rental Contract\nKNOWN ALL MEN BY THESE ...,What is the agreement end date?,20.05.2008
2,House Rental Contract\nKNOWN ALL MEN BY THESE ...,What is the renewal notice period?,15.0
3,House Rental Contract\nKNOWN ALL MEN BY THESE ...,Who is party one?,"Antonio Levy S. Ingles, Jr. and/or Mary Rose C..."
4,House Rental Contract\nKNOWN ALL MEN BY THESE ...,Who is party two?,GERALDINE Q. GALINATO


In [42]:
print(dataset[['filename', 'text']].head())

                                            filename  \
0  6683127-House-Rental-Contract-GERALDINE-GALINA...   
1  6683129-House-Rental-Contract-Geraldine-Galina...   
2                        18325926-Rental-Agreement-1   
3                          24158401-Rental-Agreement   
4                          36199312-Rental-Agreement   

                                                text  
0  House Rental Contract\nKNOWN ALL MEN BY THESE ...  
1  \n\n\n\n\nHouse Rental Contract\nKNOWN ALL MEN...  
2  \n\n\n\n\n\n\n\n\n\nRENTAL AGREEMENT\nThis dee...  
3                                                NaN  
4  RENEWAL OF RENTAL AGREEMENT\nThis AGREEMENT of...  


In [43]:
print(train_df['filename'].head())
print(documents_df['filename'].head())

0    6683127-House-Rental-Contract-GERALDINE-GALINA...
1    6683129-House-Rental-Contract-Geraldine-Galina...
2                          18325926-Rental-Agreement-1
3                            24158401-Rental-Agreement
4                            36199312-Rental-Agreement
Name: filename, dtype: object
0    6683129-House-Rental-Contract-Geraldine-Galina...
1                            54770958-Rental-Agreement
2    6683127-House-Rental-Contract-GERALDINE-GALINA...
3                            47854715-RENTAL-AGREEMENT
4                            50070534-RENTAL-AGREEMENT
Name: filename, dtype: object


In [39]:
import os

train_df["filename"] = train_df["filename"].apply(
    lambda x: os.path.basename(str(x)).strip()
)

documents_df["filename"] = documents_df["filename"].apply(
    lambda x: os.path.basename(str(x)).strip()
)

In [40]:
train_df["filename"] = train_df["filename"].apply(
    lambda x: os.path.splitext(str(x))[0]
)

documents_df["filename"] = documents_df["filename"].apply(
    lambda x: os.path.splitext(str(x))[0]
)

In [41]:
dataset = train_df.merge(
    documents_df,
    on="filename",
    how="left"
)

print(dataset["text"].isnull().sum())
display(dataset.head())

1


,filename,Aggrement Value,Aggrement Start Date,Aggrement End Date,Renewal Notice (Days),Party One,Party Two,text
0,6683127-House-Rental-Contract-GERALDINE-GALINA...,6500,20.05.2007,20.05.2008,15.0,"Antonio Levy S. Ingles, Jr. and/or Mary Rose C...",GERALDINE Q. GALINATO,House Rental Contract\nKNOWN ALL MEN BY THESE ...
1,6683129-House-Rental-Contract-Geraldine-Galina...,6500,20.05.2007,20.05.2008,15.0,"Antonio Levy S. Ingles, Jr. and/or Mary Rose C...",GERALDINE Q. GALINATO,\n\n\n\n\nHouse Rental Contract\nKNOWN ALL MEN...
2,18325926-Rental-Agreement-1,4000,05.12.2008,31.11.2009,90.0,MR.K.Kuttan,P.M. Narayana Namboodri,\n\n\n\n\n\n\n\n\n\nRENTAL AGREEMENT\nThis dee...
3,24158401-Rental-Agreement,12000,01.04.2008,31.03.2009,60.0,Hanumaiah,Vishal Bhardwaj,NaN
4,36199312-Rental-Agreement,3800,01.05.2010,31.04.2011,30.0,Balaji.R,Kartheek R,RENEWAL OF RENTAL AGREEMENT\nThis AGREEMENT of...


In [45]:
training_df = training_df.dropna(subset=["context", "answer"])

training_df = training_df[
    training_df["context"].str.len() > 0
]

training_df.reset_index(drop=True, inplace=True)

print(training_df.shape)

training_df.head()

(45, 3)


,context,question,answer
0,House Rental Contract\nKNOWN ALL MEN BY THESE ...,What is the agreement start date?,20.05.2007
1,House Rental Contract\nKNOWN ALL MEN BY THESE ...,What is the agreement end date?,20.05.2008
2,House Rental Contract\nKNOWN ALL MEN BY THESE ...,What is the renewal notice period?,15.0
3,House Rental Contract\nKNOWN ALL MEN BY THESE ...,Who is party one?,"Antonio Levy S. Ingles, Jr. and/or Mary Rose C..."
4,House Rental Contract\nKNOWN ALL MEN BY THESE ...,Who is party two?,GERALDINE Q. GALINATO


In [46]:
def find_answer_start(context, answer):

    context = str(context)
    answer = str(answer)

    start = context.lower().find(answer.lower())

    return start

In [47]:
qa_data = []

for _, row in training_df.iterrows():

    start = find_answer_start(row["context"], row["answer"])

    if start == -1:
        continue

    qa_data.append({
        "context": row["context"],
        "question": row["question"],
        "answers": {
            "text": [row["answer"]],
            "answer_start": [start]
        }
    })

print("Training Samples :", len(qa_data))

Training Samples : 12


In [48]:
from datasets import Dataset

hf_dataset = Dataset.from_list(qa_data)

hf_dataset

Dataset({
    features: ['context', 'question', 'answers'],
    num_rows: 12
})

In [49]:
from transformers import AutoTokenizer
from transformers import AutoModelForQuestionAnswering

model_name = "deepset/bert-base-cased-squad2"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForQuestionAnswering.from_pretrained(model_name)

config.json:   0%|          | 0.00/508 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/152 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/433M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForQuestionAnswering LOAD REPORT from: deepset/bert-base-cased-squad2
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [50]:
def preprocess(examples):

    return tokenizer(

        examples["question"],

        examples["context"],

        truncation=True,

        padding="max_length",

        max_length=512
    )

In [51]:
tokenized_dataset = hf_dataset.map(
    preprocess,
    batched=True
)

tokenized_dataset

Map:   0%|          | 0/12 [00:00<?, ? examples/s]

Dataset({
    features: ['context', 'question', 'answers', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 12
})

In [52]:
def preprocess_function(examples):
    inputs = tokenizer(
        examples["question"],
        examples["context"],
        max_length=512,
        truncation="only_second",
        padding="max_length",
        return_offsets_mapping=True,
    )

    offset_mapping = inputs.pop("offset_mapping")

    start_positions = []
    end_positions = []

    for i, offsets in enumerate(offset_mapping):

        answer = examples["answers"][i]

        start_char = answer["answer_start"][0]
        end_char = start_char + len(answer["text"][0])

        sequence_ids = inputs.sequence_ids(i)

        idx = 0
        while sequence_ids[idx] != 1:
            idx += 1
        context_start = idx

        while idx < len(sequence_ids) and sequence_ids[idx] == 1:
            idx += 1
        context_end = idx - 1

        if offsets[context_start][0] > end_char or offsets[context_end][1] < start_char:
            start_positions.append(0)
            end_positions.append(0)
        else:
            idx = context_start
            while idx <= context_end and offsets[idx][0] <= start_char:
                idx += 1
            start_positions.append(idx - 1)

            idx = context_end
            while idx >= context_start and offsets[idx][1] >= end_char:
                idx -= 1
            end_positions.append(idx + 1)

    inputs["start_positions"] = start_positions
    inputs["end_positions"] = end_positions

    return inputs

In [53]:
tokenized_dataset = hf_dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=hf_dataset.column_names
)

Map:   0%|          | 0/12 [00:00<?, ? examples/s]

In [54]:
tokenized_dataset.set_format(
    "torch",
    columns=[
        "input_ids",
        "attention_mask",
        "start_positions",
        "end_positions"
    ]
)

In [57]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./bert_qa",
    learning_rate=2e-5,
    per_device_train_batch_size=2,
    num_train_epochs=5
)

In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    processing_class=tokenizer
)


In [62]:
!pip uninstall -y torchvision torch torchaudio
!pip install torch torchvision torchaudio
!pip install -U datasets transformers accelerate

Found existing installation: torchvision 0.26.0+cpu
Uninstalling torchvision-0.26.0+cpu:
  Successfully uninstalled torchvision-0.26.0+cpu
Found existing installation: torch 2.11.0+cpu
Uninstalling torch-2.11.0+cpu:
  Successfully uninstalled torch-2.11.0+cpu
Found existing installation: torchaudio 2.11.0+cpu
Uninstalling torchaudio-2.11.0+cpu:
  Successfully uninstalled torchaudio-2.11.0+cpu
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 532.3/532.3 MB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.2/366.2 MB 1.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.1/170.1 MB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.0/206.0 MB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 197.7/197.7 MB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 89.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 68.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 14.2 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
^C


In [1]:
trainer.train()

NameError: name 'trainer' is not defined